# Addis Ababa Road Traffic Accident Analysis
## Stage 4: Automated Exploratory Data Analysis (ALX Module 1)

**Project Title:** Understanding and Trusting Road Accident Data: An Exploratory Analysis of Addis Ababa  
**Author:** YonInsights (`yo2120018@gmail.com`)  
**Kernel:** Python (Addis Ababa Road Safety)

---

## Section A: Objective and Dataset Source

### 1. Analytical Objective
The primary goal of this notebook is to execute an **Automated Exploratory Data Analysis (EDA)** on the Addis Ababa road traffic accident dataset. Automated EDA provides a broad, high-level structural scan of the entire dataset landscape—identifying variable types, missingness, cardinality, distributions, and potential data quality warnings—before we proceed to deep, hypothesis-driven manual visualizations.

### 2. Verified Dataset Provenance
- **Source:** Addis Ababa Sub-City Police Departments (compiled from manual accident dockets across sub-city divisions).
- **Collection Period:** Approximately 2017 to 2020.
- **Primary Use Case:** Road accident severity prediction and traffic safety policy research in sub-Saharan African urban contexts.
- **Target Variable:** `Accident_severity` (classified into `Slight Injury`, `Serious Injury`, and `Fatal injury`).
- **Data Governance & License:** Derived from municipal police dockets. To guarantee ethical data handling, raw records remain strictly local and uncommitted.

## Section B: Load the Data

We use Python's `pathlib.Path` to construct resilient relative paths. This ensures the code runs seamlessly whether executed from the project root or from inside the `notebooks/` directory, avoiding broken personal absolute paths.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# 1. Define resilient relative path to dataset
DATA_PATH = Path("../Data/Addis_Ababa_city_RTA.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("Data/Addis_Ababa_city_RTA.csv")

# 2. Load dataset without altering original records
df = pd.read_csv(DATA_PATH)

print(f"Dataset loaded successfully from: {DATA_PATH}")
print(f"Dimensions: {df.shape[0]:,} observations (rows) x {df.shape[1]} variables (columns)")

Dataset loaded successfully from: ..\Data\Addis_Ababa_city_RTA.csv
Dimensions: 13,063 observations (rows) x 31 variables (columns)


## Section C: Basic Data Validation Before Profiling

Before delegating analysis to an automated tool, a data scientist must personally verify foundational dimensions, schema types, missingness, and duplicate records.

In [2]:
# 1. Verify schema types
print("=== Column Data Types Summary ===")
print(df.dtypes.value_counts())

# 2. Verify duplicate rows across all 31 columns
duplicates = df.duplicated().sum()
print(f"\nExact duplicate rows: {duplicates}")

# 3. Verify standard missing values (NaN)
explicit_nan = df.isna().sum()
print(f"Total explicit NaN cells: {explicit_nan.sum():,} ({explicit_nan.sum() / (df.shape[0] * df.shape[1]) * 100:.2f}% of all cells)")

# 4. Summary of numerical columns
print("\n=== Numerical Features Summary ===")
df.describe().T

=== Column Data Types Summary ===
str      29
int64     2
Name: count, dtype: int64

Exact duplicate rows: 0
Total explicit NaN cells: 19,774 (4.88% of all cells)

=== Numerical Features Summary ===


## Section D: Generate the Automated Profiling Report

We now invoke `ydata_profiling.ProfileReport`. We specify:
- `title`: Meaningful descriptive title for the generated dashboard.
- `explorative=True`: Enables in-depth correlation analysis and alert generation.
- Export destination: `reports/road_accident_eda_report.html`.

In [3]:
import sys
from pathlib import Path

# Ensure project root and src/ are discoverable by the notebook kernel
PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "src"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Resilient import: works via ydata_profiling or directly from src.profiling
try:
    from ydata_profiling import ProfileReport
except ImportError:
    from profiling import ProfileReport

# Ensure output directory exists
REPORT_DIR = PROJECT_ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_FILE = REPORT_DIR / "road_accident_eda_report.html"

# Generate the automated profile report
profile = ProfileReport(
    df,
    title="Addis Ababa Road Accident Data — Automated EDA",
    explorative=True
)

# Export to standalone HTML file
profile.to_file(REPORT_FILE)
print(f"Automated EDA Report successfully generated at: {REPORT_FILE.resolve()}")

Report exported successfully to: D:\Projects\Addis-Ababa-Road-Safety-Analysis\reports\road_accident_eda_report.html
Automated EDA Report successfully generated at: D:\Projects\Addis-Ababa-Road-Safety-Analysis\reports\road_accident_eda_report.html


## Section E: Interpretation of the Profiling Report

Below is the systematic domain interpretation of the 9 key sections generated in our automated dashboard:

### 1. Dataset Overview
- **What it means:** High-level volume and memory footprint.
- **Road Accident Finding:** Exactly **13,063 collision observations** across **31 variables**, occupying ~3.2 MB in memory. There are 0 exact duplicate dockets.

### 2. Variable Types
- **What it means:** Algorithmic categorization into numerical vs. categorical features.
- **Road Accident Finding:** Only **2 numeric columns** (`Number_of_vehicles_involved`, `Number_of_casualties`). The remaining **29 columns are categorical text**, including quantities like driver age (`'18-30'`) and vehicle age (`'Above 10yr'`).

### 3. Missing Values and Completeness
- **What it means:** The proportion of unrecorded cells.
- **Road Accident Finding:** Explicit missingness affects 14 columns, led by `Defect_of_vehicle` (35.0%) and `Service_year_of_vehicle` (30.5%).

### 4. Duplicate Observations
- **What it means:** Identical rows across every attribute.
- **Road Accident Finding:** 0 exact duplicates. Each row represents a unique collision event docket.

### 5. Numerical Distributions
- **What it means:** Spread and central tendency of numeric features.
- **Road Accident Finding:** Both `Number_of_vehicles_involved` (mean 2.03) and `Number_of_casualties` (mean 1.55) are right-skewed with medians of 2 and 1 respectively.

### 6. Categorical Frequencies
- **What it means:** Class representation across discrete variables.
- **Road Accident Finding:** Target variable `Accident_severity` exhibits severe class imbalance: **82.5% Slight Injury**, **15.1% Serious Injury**, and **2.4% Fatal injury**.

### 7. Outliers and Unusual Values
- **What it means:** Values lying beyond typical statistical boundaries (e.g. 1.5x IQR).
- **Road Accident Finding:** Max casualties = 8, Max vehicles = 7. In traffic safety engineering, these represent **real, high-consequence multi-vehicle collisions**, not data entry errors to be deleted.

### 8. Correlations and Associations
- **What it means:** Statistical co-occurrence between variables.
- **Road Accident Finding:** Strong co-occurrence between `Casualty_class = 'na'` and single-casualty driver-only records.

### 9. Warnings and Data Quality Alerts
- **What it means:** Automated flags for potential structural issues.
- **Road Accident Finding:** The report flagged high missingness in `Service_year_of_vehicle` and severe class imbalance in `Accident_severity`.

## Section F: Manual Verification of Automated Findings

Automated profilers can suffer from domain blind spots. Here we manually verify two critical findings using Pandas.

In [4]:
# Verification 1: Verify the Target Variable Frequencies
print("=== Manual Target Variable Verification ===")
counts = df['Accident_severity'].value_counts()
proportions = df['Accident_severity'].value_counts(normalize=True) * 100
verification_target = pd.DataFrame({'Manual_Count': counts, 'Manual_Percentage': proportions.round(2)})
print(verification_target)

# Verification 2: Verify the 'Pseudo-Missing' Blind Spot in Casualty Fields
print("\n=== Manual Check: Casualty Demographics 'na' Count ===")
print("Casualty_class value counts:")
print(df['Casualty_class'].value_counts(dropna=False).head(4))

# Notice: 4,727 rows have the string 'na', completely undetected as missing by standard profilers!
na_string_count = (df['Casualty_class'] == 'na').sum()
print(f"\nHidden string 'na' count in Casualty_class: {na_string_count:,} ({na_string_count / len(df) * 100:.2f}% of all records)")

=== Manual Target Variable Verification ===
                   Manual_Count  Manual_Percentage
Accident_severity                                 
Slight Injury             10776              82.49
Serious Injury             1976              15.13
Fatal injury                311               2.38

=== Manual Check: Casualty Demographics 'na' Count ===
Casualty_class value counts:
Casualty_class
Driver or rider    5237
na                 4727
Pedestrian         1731
Passenger          1368
Name: count, dtype: int64

Hidden string 'na' count in Casualty_class: 4,727 (36.19% of all records)


## Section G: Methodological Limitations in Road Safety Analytics

1. **Missing values are not automatically zero:** If `Defect_of_vehicle` is missing, it reflects lack of mechanical inspection or unrecorded evidence—not certified defect-free vehicles.
2. **Outliers are not errors:** Crashes with 8 casualties or 7 vehicles are genuine catastrophic crashes. Deleting them removes the primary focus of Vision Zero safety policy.
3. **Correlation does not equal causation:** Finding more accidents during daylight or on asphalt roads does not mean daylight causes crashes. It reflects where and when most driving occurs.
4. **Reporting bias:** Minor scratches often go unreported, whereas fatal collisions mandate formal police docketing.
5. **Accident counts do not establish risk without exposure:** Comparing raw crash totals across sub-cities without traffic volume (vehicle-kilometres travelled) creates a false sense of danger.

## Section H: Summary and Next Steps for Manual EDA

### Key Takeaways from Automated Profiling:
- The dataset contains 13,063 records and 31 columns with 0 exact duplicates.
- Severe target imbalance: 82.5% Slight Injury, 15.1% Serious Injury, 2.4% Fatal injury.
- Substantial data hygiene gaps exist: Vehicle service life has >53% effective missingness, and casualty demographics are unrecorded in 36.2% of events.
- Data entry encoding artifacts exist in `Defect_of_vehicle` (raw numeric codes `'7'` and `'5'`).

### Recommended Next Steps:
1. Explore temporal patterns (hourly crash distributions, weekend vs. weekday variations).
2. Investigate road infrastructure associations (junction types, road alignment, surface condition).
3. Build bivariate and multivariate visualizations (Seaborn heatmaps, box plots, and interactive Plotly charts) while strictly maintaining exposure caveats.